# Arrow filesystems - Python

All 5 Python examples from [docs/arrowfs.md](https://platob.github.io/yggdryl/arrowfs/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and run on any Python 3 kernel with the package
installed:

```console
pip install yggdryl
```

## Construct from a filesystem and a path

In [ ]:
import tempfile, pathlib
import pyarrow.fs as pafs
from yggdryl import IOBase

root = pathlib.Path(tempfile.mkdtemp())
handle = IOBase.from_arrow_fs(pafs.LocalFileSystem(), (root / "trades.bin").as_posix())

# Per the laziness contract nothing exists until something is written.
assert not handle.exists()
assert handle.read_bytes() == b""

with handle:
    handle.write_bytes(b"AAPL")

assert handle.read_bytes() == b"AAPL"
assert (root / "trades.bin").read_bytes() == b"AAPL"

## A write publishes when the handle closes

In [ ]:
import tempfile, pathlib
import pyarrow.fs as pafs
from yggdryl import IOBase

root = pathlib.Path(tempfile.mkdtemp())
handle = IOBase.from_arrow_fs(pafs.LocalFileSystem(), (root / "staged.bin").as_posix())

handle.write_bytes(b"pending")
assert not (root / "staged.bin").exists()

handle.close()
assert (root / "staged.bin").read_bytes() == b"pending"

## Folders, globs, and partitions

In [ ]:
import tempfile, pathlib
import pyarrow.fs as pafs
from yggdryl import IOBase

root = pathlib.Path(tempfile.mkdtemp()) / "lake"
for year in ("2024", "2025"):
    leaf = root / f"year={year}"
    leaf.mkdir(parents=True)
    (leaf / "part-0.parquet").write_bytes(b"PAR1")

lake = IOBase.from_arrow_fs(pafs.LocalFileSystem(), root.as_posix())

assert lake.is_dir()
assert len(lake.iterdir()) == 2
assert len(lake.glob("**/*.parquet")) == 2
assert len(lake.children_where({"year": "2024"})) == 1

# A child still carries the filesystem it came from.
part = lake / "year=2024" / "part-0.parquet"
assert part.read_bytes() == b"PAR1"

## Records

In [ ]:
import tempfile, pathlib
import pyarrow as pa
import pyarrow.fs as pafs
import pyarrow.parquet as pq
from yggdryl import IOBase

root = pathlib.Path(tempfile.mkdtemp())
table = pa.table({"id": [1, 2], "symbol": ["AAPL", "MSFT"]})

handle = IOBase.from_arrow_fs(pafs.LocalFileSystem(), (root / "trades.parquet").as_posix())
with handle:
    handle.write_arrow_batch_reader(table)

assert handle.read_arrow_batch_reader().read_all().num_rows == 2

# What landed is an ordinary Parquet file, so PyArrow reads it back.
assert pq.read_table(root / "trades.parquet").equals(table)

## Composing with the wrappers

In [ ]:
import tempfile, pathlib
import pyarrow as pa
import pyarrow.fs as pafs
from yggdryl import IOBase, iceberg

root = pathlib.Path(tempfile.mkdtemp())
table_rows = pa.table({"id": [1, 2], "symbol": ["AAPL", "MSFT"]})

warehouse = IOBase.from_arrow_fs(pafs.LocalFileSystem(), (root / "trades").as_posix())
table = iceberg.Table.create(warehouse, table_rows.schema)
table.append(table_rows)

assert table.scan().read_all().num_rows == 2